# Laya fine-tune: train, compare, report (one notebook, press **Run all**)

What it does, in order: checks the GPU, mounts Drive, installs the pinned libraries, downloads the untouched English checkpoint, scores the **base model** on the dev and test rows, trains two variants (soft-ce and rlcd), scores both, picks the better one on **dev**, saves everything to Drive, and prints a short summary to paste back.

Expect roughly 30 to 40 minutes on an L4. The data is embedded in this notebook (nothing to upload).

**Read this before trusting any number:** every train, dev and test row was written *and labelled by Claude*. No row is human-labelled and the test set is not independent. The test rows use industries, personas and wording that the training rows never contain, so the test gap is a fair sign of generalisation, but the scores say "good on Claude-written synthetic calls", not "good on real calls".

If a cell fails, the error is printed under it. Paste that to Claude. After a kernel restart: run cell 1, then continue from the cell that failed.

Nothing here modifies the original checkpoint: training reads a downloaded copy and writes new folders.

In [ ]:
# CELL 1: settings + GPU check.
# EXPECT: "NVIDIA L4, 23034 MiB" (about 23 GB), "cuda True", then the run id.
import datetime
import hashlib
import json
import os
import queue
import re
import shutil
import subprocess
import sys
import threading
import time

import torch

BASE_REPO = "convaiinnovations/laya"
BASE_REV = "7b928d828b7b0e022f929d9bd2e44165aa270148"      # the snapshot used locally (laya-test/hf_home)
LAYA_COMMIT = "3cf26cbcb18725dbc2d127bb8bb2c4c43243ae63"   # laya v0.4.0
LOCAL_BASE_SHA256 = {
    "model.safetensors": "891102d372688fc2a094dac56a384bc537b87c63f21f9f3dac0be2b7cbc8d86c",
    "rl_agent_config.json": "ae287b56bbcf5f8c4f4541ae9dfd00c914c4c48b940b8398c3058af37ba92bbd",
}
TRAINED = ["rapport", "paraphrasing", "active_listening", "resolved", "empathy"]    # checks this fine-tune is meant to improve
REGRESSION = ["ownership", "probing", "hostility", "escalation"]                     # checks that must not get worse

DRIVE = "/content/drive/MyDrive/laya-finetuning"
WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)

# one run id per session, kept on disk so re-running this cell does not change it
_rid = f"{WORK}/run_id.txt"
if not os.path.exists(_rid):
    open(_rid, "w").write("ft1-" + datetime.datetime.now().strftime("%Y%m%d-%H%M"))
RUN_ID = open(_rid).read().strip()

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip())
print("cuda", torch.cuda.is_available(), "| torch", torch.__version__)
print("run id:", RUN_ID)

In [ ]:
# CELL 2: mount Google Drive and create the folders.
# EXPECT: Colab asks you to allow Drive access, then "Mounted at /content/drive" and "DRIVE_OK [...]".
# If it prints DRIVE_MOUNT_FAILED, stop and paste the message.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    for sub in ("checkpoints", "data", "runs"):
        os.makedirs(f"{DRIVE}/{sub}", exist_ok=True)
    print("DRIVE_OK", sorted(os.listdir(DRIVE)))
except Exception as e:
    print("DRIVE_MOUNT_FAILED:", repr(e))
    raise

In [ ]:
# CELL 3: install laya (same commit as the local v0.4.0 checkout) and the same transformers as local.
# EXPECT: no errors, then "laya 0.4.0" and "transformers 5.16.1".
# If pip says a restart is needed: restart the kernel, re-run cell 1, SKIP this cell, continue at cell 4.
!pip -q install "laya @ git+https://github.com/NandhaKishorM/laya.git@{LAYA_COMMIT}" "transformers==5.16.1"
from importlib.metadata import version
print("laya", version("laya"), "| transformers", version("transformers"), "| torch", version("torch"))

In [ ]:
# CELL 4: unpack the embedded data and the frozen production questions, and check their hashes.
# EXPECT: "data sha OK" for train/dev/test, "frozen checks sha OK", row counts (train 1269, dev 273, test 396),
# and a table of rows per check. Every row must carry the same inputs hash as the frozen questions.
import base64
import gzip

FROZEN = json.loads(r"""{
  "frozen": {
    "checks": {
      "empathy": {
        "yes_means": "PASS",
        "pass_threshold": 0.78,
        "question": {
          "type": "noul",
          "instructions": "Does the agent acknowledge the customer's frustration with empathy?",
          "criteria": {
            "true": "the agent acknowledges the customer's frustration or feelings and responds with care",
            "false": "the agent ignores, brushes off or is rude about the customer's frustration"
          },
          "labels": {
            "true": "A",
            "false": "B"
          }
        }
      },
      "active_listening": {
        "yes_means": "FAIL",
        "pass_threshold": 0.19,
        "question": {
          "type": "noul",
          "instructions": "Does the second agent question ask again for information the customer already gave in their answer?",
          "criteria": {
            "true": "the second question asks again for information the customer already gave in their answer",
            "false": "the second question asks for something new, or re-checks something that may have changed"
          }
        }
      },
      "ownership": {
        "yes_means": "FAIL",
        "pass_threshold": 0.52,
        "question": {
          "type": "noul",
          "instructions": "Does the agent deflect responsibility, blame another team, or refuse to help?",
          "criteria": {
            "true": "the agent deflects, blames another team or department, refuses to help, or tells the customer to contact someone else",
            "false": "the agent takes ownership and keeps working on the customer's problem"
          },
          "labels": {
            "true": "A",
            "false": "B"
          }
        }
      },
      "rapport": {
        "yes_means": "FAIL",
        "pass_threshold": 0.5,
        "question": {
          "type": "noul",
          "instructions": "Is any agent line rude, condescending, dismissive, sarcastic or unprofessional toward the customer?",
          "criteria": {
            "true": "at least one agent line is rude, condescending, dismissive, sarcastic or unprofessional toward the customer, for example blaming the customer, talking down to them or brushing them off",
            "false": "every agent line is polite, respectful and professional"
          }
        }
      },
      "probing": {
        "yes_means": "PASS",
        "pass_threshold": 0.7,
        "question": {
          "type": "noul",
          "instructions": "Did the agent ask at least one genuine diagnostic question about the customer's problem?",
          "criteria": {
            "true": "the agent asked at least one question that investigates the cause or specifics of the problem, such as when it started, which device, what error or lights, or what was already tried",
            "false": "the agent only asked greetings, identity verification, confirmations, upsell, scheduling or closing questions"
          },
          "labels": {
            "true": "A",
            "false": "B"
          }
        }
      },
      "resolved": {
        "yes_means": "PASS",
        "pass_threshold": 0.73,
        "question": {
          "type": "noul",
          "instructions": "Did the agent explicitly confirm that the customer's issue was resolved?",
          "criteria": {
            "true": "the agent explicitly confirms the issue is resolved or fixed, usually after checking with the customer",
            "false": "the agent never confirms resolution: the call just ends, the agent assumes it is fixed, the customer says it still fails, or it is handed off"
          },
          "labels": {
            "true": "A",
            "false": "B"
          }
        }
      },
      "escalation": {
        "yes_means": "FAIL",
        "pass_threshold": 0.47,
        "question": {
          "type": "noul",
          "instructions": "Did the customer ask for a supervisor, manager or escalation that the agent refused, ignored or discouraged?",
          "criteria": {
            "true": "the customer asked for a supervisor, manager or escalation and the agent refused, ignored, discouraged or faked it",
            "false": "the customer did not ask to escalate, or the agent handled the request properly"
          }
        }
      },
      "hostility": {
        "yes_means": "FAIL",
        "pass_threshold": 0.79,
        "question": {
          "type": "noul",
          "instructions": "Is the agent hostile, abusive, insulting, threatening, mocking, or hangs up on the customer?",
          "criteria": {
            "true": "the agent insults, mocks, threatens, is abusive toward the customer, or abruptly hangs up",
            "false": "the agent stays professional and respectful, even if the customer is angry"
          }
        }
      },
      "paraphrasing": {
        "yes_means": "PASS",
        "pass_threshold": 0.5,
        "question": {
          "type": "noul",
          "instructions": "Does the agent restate or paraphrase the customer's problem back to them?",
          "criteria": {
            "true": "the agent restates the customer's problem in their own words to confirm they understood it",
            "false": "the agent never restates the customer's problem"
          }
        }
      }
    },
    "caps": {
      "EMPATHY_MAX_SNIPPETS": 2,
      "PROBING_PROBLEM_MAX_WORDS": 50,
      "PROBING_QUESTIONS_MAX_WORDS": 100,
      "RESOLVED_TAIL_LINES": 10,
      "PARAPHRASE_CUSTOMER_MAX_WORDS": 50,
      "PARAPHRASE_AGENT_MAX_WORDS": 150
    }
  },
  "sha256": "f31b14a509629d6554551702f4d6c50fd6fa41a694186b7b87d2843a78456f61",
  "inputs_sha256": "66deff95a20ce04e2b731dc0f7fea374761f7eca36337f88bed6ff0bcf582c2e",
  "extractor_sources_sha256": {
    "laya_inputs.py": "611d73145101249872eadd0a6fbe9491d26fb6b90eca3832e84fa877e60c969c",
    "rule_engine.py": "5b755cea212a2dd409db3e01f64ec29ef50ab94c5e3378cdb2ea5041bc13bce9"
  },
  "n_questions": 9,
  "qa_analysis_commit": "53becd834c0ae76c4ab19702f785483c109d28b0",
  "qa_analysis_branch": "version-10.2-laya-finetune",
  "qa_analysis_dirty": false
}""")
DATA_B64 = {"train_all": "...", "dev_all": "...", "test": "..."}
DATA_SHA256 = {"train_all": "556389dc5e8802d5ed4e2d2ab82c89d66688484c05a755c5a079afbbf353b37f", "dev_all": "cde28dbd1cd863defb01eef19c8f9ae10f3dfac68a571c78f20344e8eb345cc1", "test": "c8334ea87b5a12a0ee05a09feb82893856032c1abd48ee0340ef025e2404fd23"}


def canonical_sha(obj):
    text = json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(",", ":"))
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


assert canonical_sha(FROZEN["frozen"]) == FROZEN["sha256"], "frozen checks were altered"
CHECKS = FROZEN["frozen"]["checks"]
print("frozen checks sha OK:", FROZEN["sha256"][:12] + "...", "|", len(CHECKS), "questions")

DATA_DIR = f"{WORK}/data"
os.makedirs(DATA_DIR, exist_ok=True)
SPLIT = {}
for name, b64 in DATA_B64.items():
    raw = gzip.decompress(base64.b64decode(b64))
    assert hashlib.sha256(raw).hexdigest() == DATA_SHA256[name], f"{name}: embedded data was altered"
    open(f"{DATA_DIR}/{name}.jsonl", "wb").write(raw)
    SPLIT[name] = [json.loads(l) for l in raw.decode("utf-8").splitlines() if l.strip()]
    print(f"data sha OK: {name:<10} {len(SPLIT[name])} rows")
for s in SPLIT.values():
    assert all(r["meta"]["inputs_sha256"] == FROZEN["inputs_sha256"] for r in s), "rows were built from different inputs"

import collections
tbl = collections.defaultdict(lambda: collections.Counter())
for name, rows in SPLIT.items():
    for r in rows:
        tbl[r["meta"]["check"]][name] += 1
print(f"{'check':<18}" + "".join(f"{n:>10}" for n in SPLIT))
for c, cnt in tbl.items():
    print(f"{c:<18}" + "".join(f"{cnt[n]:>10}" for n in SPLIT))

In [ ]:
# CELL 5: download the English checkpoint at the pinned revision and compare it with the local copy.
# EXPECT: files listed (model.safetensors about 842 MB), "weights identical to local copy: True",
# "config identical to local copy: True", and max_len 512 / head_max_len 192.
from huggingface_hub import snapshot_download


def sha256_file(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()


BASE = snapshot_download(BASE_REPO, revision=BASE_REV, local_dir=f"{WORK}/base-english",
                         allow_patterns=["model.safetensors", "rl_agent_config.json", "encoder/*", "tokenizer/*"])
for dp, _, fns in sorted(os.walk(BASE)):
    for fn in sorted(fns):
        p = os.path.join(dp, fn)
        if "/.cache" not in p.replace("\\", "/"):
            print(f"{os.path.getsize(p) / 1e6:9.1f} MB  {os.path.relpath(p, BASE)}")

BASE_SHA256 = {name: sha256_file(f"{BASE}/{name}") for name in LOCAL_BASE_SHA256}
same_w = BASE_SHA256["model.safetensors"] == LOCAL_BASE_SHA256["model.safetensors"]
same_c = BASE_SHA256["rl_agent_config.json"] == LOCAL_BASE_SHA256["rl_agent_config.json"]
print("weights identical to local copy:", same_w)
print("config identical to local copy:", same_c)
BASE_CFG = json.load(open(f"{BASE}/rl_agent_config.json"))
print("max_len", BASE_CFG["max_len"], "| head_max_len", BASE_CFG["head_max_len"], "| encoder", BASE_CFG["encoder"])
assert same_w and same_c, "downloaded base differs from the local snapshot: stop and tell Claude"

In [ ]:
# CELL 6: helpers for safe copying to Drive and for running laya-train with live output (also writes the GPU-fix launcher).
# Also writes laya_train_cuda.py: laya 0.4.0's trainer crashes on a GPU ("tensors on the same device ...") because it
# evaluates before moving the model to the GPU. This wrapper does that move first; the package itself is not edited.
# EXPECT: no output other than "helpers ready".
LAUNCHER = f"{WORK}/laya_train_cuda.py"
with open(LAUNCHER, "w", encoding="utf-8", newline="\n") as f:
    f.write(r'''"""laya-train with one fix for GPUs. Same arguments as `laya-train` / `python -m laya.train_cli`.

Why it exists: in laya 0.4.0, finetune() evaluates the calibration slice (or the --eval file) BEFORE training,
but load_checkpoint() returns the model on the CPU and only train_model() moves it to the GPU. On CUDA that
evaluation crashes with "Expected all tensors to be on the same device ... index_select". Laya's own tests run
on CPU, so they never hit it. This wrapper moves the model to the target device before every
calibration_records() call. Nothing in the installed laya package is edited.

Use it exactly like laya-train:
    python laya_train_cuda.py --data train.jsonl --base <dir> --out <dir> --loss soft-ce --epochs 4 ...
Drop this wrapper if a later laya release fixes it upstream (run once without it to find out).
"""
import sys

import laya.train as _train

_original = _train.calibration_records


def _calibration_records_on_device(model, tok, items, device, *args, **kwargs):
    model.to(device)      # a no-op when it is already there (after training)
    return _original(model, tok, items, device, *args, **kwargs)


# finetune() looks calibration_records up in laya.train's globals at call time, so this takes effect
_train.calibration_records = _calibration_records_on_device

from laya.train_cli import main  # noqa: E402  (imported after the patch on purpose)

if __name__ == "__main__":
    sys.exit(main())
''')

EPOCH_RE = re.compile(r"epoch (\d+)/(\d+) mean loss")


class SourceChanged(Exception):
    pass


class CopyMismatch(Exception):
    pass


def tree_stats(path, ignore=()):
    """Sorted (relative path, size, mtime_ns) for every file under path. Files that vanish mid-scan are skipped."""
    out = []
    for dp, dns, fns in os.walk(path):
        if dp == path:
            dns[:] = [d for d in dns if d not in ignore]
        for fn in fns:
            p = os.path.join(dp, fn)
            try:
                st = os.stat(p)
            except FileNotFoundError:
                continue
            out.append((os.path.relpath(p, path), st.st_size, st.st_mtime_ns))
    return sorted(out)


def wait_stable(path, quiet_s=30, poll_s=3, timeout_s=1800, newer_than=None, ignore=()):
    """Return once no file under path changed size or mtime for quiet_s seconds.

    With newer_than (a time.time() value), also require that something was written at or after it, so an
    older, already-stable copy is never mistaken for the checkpoint that is about to be written.
    """
    t0, last, since = time.time(), None, time.time()
    while time.time() - t0 < timeout_s:
        snap = tree_stats(path, ignore)
        if snap != last:
            last, since = snap, time.time()
        fresh = newer_than is None or (snap and max(m for _, _, m in snap) / 1e9 >= newer_than)
        if snap and fresh and time.time() - since >= quiet_s:
            return snap
        time.sleep(poll_s)
    raise TimeoutError(f"{path} was not stable within {timeout_s}s")


def copy_verified(src, dst, tag, ignore=()):
    """Copy src to a temp folder next to dst, verify it, then rename it into place.

    Raises SourceChanged if src changed while copying, CopyMismatch if the copy differs. dst is only
    replaced after both checks pass, so a half-written copy never sits under the final name.
    Returns (number of files, total bytes).
    """
    tmp = f"{dst}.tmp-{tag}"
    if os.path.exists(tmp):
        shutil.rmtree(tmp)
    before = tree_stats(src, ignore)
    shutil.copytree(src, tmp, ignore=shutil.ignore_patterns(*ignore) if ignore else None)
    names_sizes = lambda stats: [(r, z) for r, z, _ in stats]
    if names_sizes(tree_stats(tmp)) != names_sizes(before):
        shutil.rmtree(tmp, ignore_errors=True)
        raise CopyMismatch(f"{tmp} differs from {src}")
    if tree_stats(src, ignore) != before:
        shutil.rmtree(tmp, ignore_errors=True)
        raise SourceChanged(src)
    old = None
    if os.path.exists(dst):
        old = f"{dst}.old-{tag}"
        os.rename(dst, old)
    os.rename(tmp, dst)
    if old:
        shutil.rmtree(old, ignore_errors=True)
    return len(before), sum(z for _, z, _ in before)


class CkptWatcher:
    """Copies src (checkpoint_latest/) to Drive each time an epoch finishes AND the folder has stopped changing.

    laya-train prints the "epoch N/M mean loss" line BEFORE it writes the checkpoint, so the line alone is
    not enough: after it, wait until nothing was written for quiet_s seconds (and something was written
    after the line), copy to a temp folder on Drive, verify, rename. finish() does one last pass after the
    process exits, because laya-train rewrites the config after calibration.
    Messages are queued and printed by the notebook's main thread (flush), never from this thread.
    """

    def __init__(self, src, dst, quiet_s=30):
        self.src, self.dst, self.quiet_s = src, dst, quiet_s
        self.work, self.msgs = queue.Queue(), queue.Queue()
        self.copies, self.errors = [], []
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()

    def say(self, text):
        self.msgs.put(f"[watcher {time.strftime('%H:%M:%S')}] {text}")

    def feed(self, item):
        t, line = item
        m = EPOCH_RE.search(line)
        if m:
            self.work.put(("epoch", int(m.group(1)), t))

    def finish(self):
        self.work.put(("final", 0, time.time()))

    def stop(self):
        """Training failed: end the thread without waiting for a checkpoint that may never come."""
        self.work.put(("stop", 0, time.time()))

    def flush(self):
        while not self.msgs.empty():
            print(self.msgs.get())

    def _loop(self):
        while True:
            kind, n, t = self.work.get()
            if kind == "stop":
                self.say("training failed, nothing more to copy")
                return
            if kind == "final" and not os.path.isdir(self.src):
                self.say("no checkpoint_latest was written, nothing to copy")
                return
            tag = f"epoch{n}" if kind == "epoch" else "final"
            quiet = self.quiet_s if kind == "epoch" else 10
            try:
                for attempt in range(1, 4):
                    wait_stable(self.src, quiet_s=quiet, newer_than=(t - 2) if kind == "epoch" else None)
                    try:
                        files, size = copy_verified(self.src, self.dst, tag)
                    except SourceChanged:
                        self.say(f"{tag}: folder changed while copying, retry {attempt}/3")
                        continue
                    self.say(f"{tag}: quiet for {quiet}s -> copied to tmp -> verified {files} files / "
                             f"{size / 1e6:.0f} MB -> renamed to {os.path.basename(self.dst)}")
                    self.copies.append((tag, files, size))
                    break
                else:
                    raise RuntimeError(f"{tag}: source kept changing")
            except Exception as e:  # reported by the notebook; never kills training
                self.errors.append(f"{tag}: {e!r}")
                self.say(f"ERROR {tag}: {e!r}")
            if kind == "final":
                return


def run_logged(cmd, log_path, on_item=None):
    """Run cmd, echo its output live, save it to log_path. on_item((time, line)) sees every line; on_item(None) is a tick.

    Returns (return code, [(time, line), ...]).
    """
    q, events = queue.Queue(), []
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

    def reader():
        for line in proc.stdout:
            q.put((time.time(), line.rstrip("\n")))
        q.put(None)

    threading.Thread(target=reader, daemon=True).start()
    with open(log_path, "w", encoding="utf-8") as log:
        while True:
            try:
                item = q.get(timeout=1)
            except queue.Empty:
                if on_item:
                    on_item(None)
                continue
            if item is None:
                break
            events.append(item)
            log.write(item[1] + "\n")
            log.flush()
            print(item[1])
            if on_item:
                on_item(item)
    return proc.wait(), events


print("helpers ready")

In [ ]:
# CELL 7: scoring and metric helpers (balanced accuracy, AUC, bootstrap intervals by pair, ECE, replay drift).
# EXPECT: only "eval helpers ready".
import random

import laya
import numpy as np


def predict_rows(agent, rows):
    """{"<row id>/<question id>": P(yes)} for every question of every row."""
    out = {}
    for r in rows:
        res = agent.predict(r["state"], r["questions"])
        for qid in r["questions"]:
            out[f"{r['meta']['id']}/{qid}"] = float(res["answers"][qid]["noul"])
    return out


def p_pass(check, p_yes):
    return p_yes if CHECKS[check]["yes_means"] == "PASS" else 1.0 - p_yes


def supervised(rows):
    return [r for r in rows if r["meta"]["check"] != "replay"]


def by_check(rows, preds):
    """{check: [(P(pass), is_pass, pair_id), ...]}"""
    d = {}
    for r in supervised(rows):
        c = r["meta"]["check"]
        d.setdefault(c, []).append((p_pass(c, preds[f"{r['meta']['id']}/{c}"]), r["meta"]["label"] == "PASS", r["meta"]["pair_id"]))
    return d


def bal_acc(items, thr):
    pos = [p for p, y, _ in items if y]
    neg = [p for p, y, _ in items if not y]
    if not pos or not neg:
        return float("nan")
    return 0.5 * (sum(p >= thr for p in pos) / len(pos) + sum(p < thr for p in neg) / len(neg))


def auc(items):
    pos = [p for p, y, _ in items if y]
    neg = [p for p, y, _ in items if not y]
    if not pos or not neg:
        return float("nan")
    wins = sum((a > b) + 0.5 * (a == b) for a in pos for b in neg)
    return wins / (len(pos) * len(neg))


def best_cut(items):
    """Cutoff with the best balanced accuracy on these items; ties go to the one nearest 0.5."""
    ps = sorted({p for p, _, _ in items})
    cands = [0.0] + [(a + b) / 2 for a, b in zip(ps, ps[1:])] + [1.0]
    scored = [(bal_acc(items, t), -abs(t - 0.5), t) for t in cands]
    best = max(scored)
    return best[2], best[0]


def boot_ci(items, thr, n=1000, seed=0):
    """95% interval of balanced accuracy, resampling PAIRS (a pair's two rows always travel together)."""
    groups = {}
    for it in items:
        groups.setdefault(it[2], []).append(it)
    keys = list(groups)
    rng = random.Random(seed)
    vals = []
    for _ in range(n):
        sample = [it for k in (rng.choice(keys) for _ in keys) for it in groups[k]]
        v = bal_acc(sample, thr)
        if v == v:
            vals.append(v)
    vals.sort()
    return vals[int(0.025 * len(vals))], vals[int(0.975 * len(vals)) - 1]


def ece(rows, preds, bins=10):
    """Expected calibration error of P(yes) against the yes/no truth, pooled over all supervised rows."""
    pts = [(preds[f"{r['meta']['id']}/{r['meta']['check']}"], bool(r["expected"][r["meta"]["check"]])) for r in supervised(rows)]
    total, err = len(pts), 0.0
    for b in range(bins):
        lo, hi = b / bins, (b + 1) / bins
        sel = [(p, y) for p, y in pts if lo <= p < hi or (b == bins - 1 and p == 1.0)]
        if sel:
            err += len(sel) / total * abs(sum(p for p, _ in sel) / len(sel) - sum(y for _, y in sel) / len(sel))
    return err


def replay_drift(rows, preds):
    """How far the tuned model's answers to generic custom-criteria questions moved from the base model's own."""
    diffs, agree, n = [], 0, 0
    for r in rows:
        if r["meta"]["check"] != "replay":
            continue
        for qid, g in r["gold"].items():
            base_p = g["probabilities"]["true"]
            p = preds[f"{r['meta']['id']}/{qid}"]
            diffs.append(abs(p - base_p))
            agree += (p > 0.5) == (base_p > 0.5)
            n += 1
    return {"items": n, "mean_abs_diff": float(np.mean(diffs)), "verdict_agreement": agree / n}


def evaluate(dev_rows, dev_preds, test_rows, test_preds, cuts=None):
    """Per-check metrics on test. Cutoffs for the TRAINED checks are tuned on dev (or passed in); regression checks keep the frozen cutoff."""
    dv, ts = by_check(dev_rows, dev_preds), by_check(test_rows, test_preds)
    out = {"checks": {}, "ece_test": ece(test_rows, test_preds), "replay_dev": replay_drift(dev_rows, dev_preds)}
    for c in TRAINED + REGRESSION:
        frozen_cut = CHECKS[c]["pass_threshold"]
        row = {"n_test": len(ts[c]), "frozen_cut": frozen_cut, "auc_test": auc(ts[c]), "auc_dev": auc(dv[c]),
               "ba_test_frozen_cut": bal_acc(ts[c], frozen_cut), "ba_dev_frozen_cut": bal_acc(dv[c], frozen_cut)}
        if c in TRAINED:
            cut = cuts[c] if cuts else best_cut(dv[c])[0]
            row.update({"tuned_cut": cut, "ba_dev_tuned": bal_acc(dv[c], cut), "ba_test_tuned": bal_acc(ts[c], cut)})
            row["ba_test_tuned_ci"] = boot_ci(ts[c], cut)
        else:
            row["ba_test_frozen_ci"] = boot_ci(ts[c], frozen_cut)
        out["checks"][c] = row
    out["mean_auc_dev_trained"] = float(np.mean([out["checks"][c]["auc_dev"] for c in TRAINED]))
    out["mean_ba_test_trained"] = float(np.mean([out["checks"][c]["ba_test_tuned"] for c in TRAINED]))
    return out


DEV, TEST = SPLIT["dev_all"], SPLIT["test"]
print("eval helpers ready")

In [ ]:
# CELL 8: baseline. The untouched base model answers every dev and test question (about 1 minute).
# EXPECT: a table of the BASE model's test scores per check: balanced accuracy at the production cutoff and AUC.
# AUC 0.50 = no signal; the base model is expected to be weak on rapport. Saved to /content/work/preds_base.json.
t0 = time.time()
base_agent = laya.load(BASE)
PRED = {"base": {"dev": predict_rows(base_agent, DEV), "test": predict_rows(base_agent, TEST)}}
del base_agent
torch.cuda.empty_cache()
json.dump(PRED["base"], open(f"{WORK}/preds_base.json", "w"))
RES = {"base": evaluate(DEV, PRED["base"]["dev"], TEST, PRED["base"]["test"])}
print(f"base scored in {time.time() - t0:.0f}s")
print(f"{'check':<18}{'n':>5}{'BA@prod cut':>13}{'AUC':>7}")
for c in TRAINED + REGRESSION:
    r = RES["base"]["checks"][c]
    print(f"{c:<18}{r['n_test']:>5}{r['ba_test_frozen_cut']:>13.3f}{r['auc_test']:>7.3f}")
print("ECE (test):", round(RES["base"]["ece_test"], 3))

In [ ]:
# CELL 9: train two variants on train_all (soft-ce and rlcd), 4 epochs each, and score each on dev and test.
# laya-train writes to /content (local disk); a watcher copies checkpoint_latest/ to Drive once it is fully written.
# EXPECT per variant (about 7-10 min, ~1,270 rows x 4 epochs at roughly 14 rows/s): "train items ..., calibration items ...",
# "epoch N/4 mean loss ..." with the loss falling, "Training complete!", watcher lines, then "exit code 0".
# A RuntimeWarning about calibration is fine. If one variant fails the other still runs; paste the error.
RUNS = [("softce", ["--loss", "soft-ce", "--label-smoothing", "0.05"]), ("rlcd", ["--loss", "rlcd"])]
COMMON = ["--epochs", "4", "--micro-batch", "8", "--grad-accum", "2", "--seed", "0", "--max-len", "512", "--head-max-len", "192"]
DRIVE_RUN = f"{DRIVE}/checkpoints/{RUN_ID}"
os.makedirs(DRIVE_RUN, exist_ok=True)

# dry run first: prints how many items laya-train builds and skips (should skip none)
rc, ev = run_logged([sys.executable, "-u", LAUNCHER, "--data", f"{DATA_DIR}/train_all.jsonl", "--base", BASE, "--dry-run",
                     "--max-len", "512", "--head-max-len", "192"], f"{WORK}/dry_run.log")
assert rc == 0, "dry run failed: paste the output above"

TRAIN_INFO = {}
for name, loss_args in RUNS:
    OUT = f"{WORK}/ckpt-{name}"
    shutil.rmtree(OUT, ignore_errors=True)
    cmd = [sys.executable, "-u", LAUNCHER, "--data", f"{DATA_DIR}/train_all.jsonl", "--base", BASE, "--out", OUT] + loss_args + COMMON
    print(f"\n===== {name}: {' '.join(cmd[3:])}")
    watcher = CkptWatcher(f"{OUT}/checkpoint_latest", f"{DRIVE_RUN}/checkpoint_latest-{name}", quiet_s=30)
    t0 = time.time()
    rc, events = run_logged(cmd, f"{WORK}/train_{name}.log", on_item=lambda it: ((watcher.feed(it) if it else None), watcher.flush()))
    (watcher.finish if rc == 0 else watcher.stop)()
    watcher.thread.join(timeout=300)
    watcher.flush()
    wall = time.time() - t0
    print(f"{name}: exit code {rc} | wall {wall:.0f}s | watcher copies {len(watcher.copies)} | watcher errors {watcher.errors}")
    TRAIN_INFO[name] = {"rc": rc, "wall_s": round(wall), "cmd": " ".join(cmd), "watcher_errors": watcher.errors,
                        "losses": [l for _, l in events if "mean loss" in l]}
    if rc != 0:
        print(f"{name} FAILED: skipping it, the error is in the output above")
        continue
    t1 = time.time()
    agent = laya.load(OUT)
    PRED[name] = {"dev": predict_rows(agent, DEV), "test": predict_rows(agent, TEST)}
    del agent
    torch.cuda.empty_cache()
    json.dump(PRED[name], open(f"{WORK}/preds_{name}.json", "w"))
    RES[name] = evaluate(DEV, PRED[name]["dev"], TEST, PRED[name]["test"])
    print(f"{name}: scored in {time.time() - t1:.0f}s | mean dev AUC (trained checks) {RES[name]['mean_auc_dev_trained']:.3f}")
assert any(n in RES for n, _ in RUNS), "no variant trained successfully: paste the errors above"

In [ ]:
# CELL 10: compare base vs tuned, pick the better variant on DEV, print the tables.
# EXPECT: WINNER line, then one table per variant with test balanced accuracy (BA) with a 95% interval (resampled by pair),
# AUC, tuned cutoff (from dev) and base BA; then replay drift and calibration error.
# Reading it: BA 0.50 = coin flip, 1.00 = perfect. AUC is cutoff-free (does the model rank PASS above FAIL?).
# Regression checks are only compared with the base model at the production cutoff; they should not drop.
VARIANTS = [n for n, _ in RUNS if n in RES]
WINNER = max(VARIANTS, key=lambda n: RES[n]["mean_auc_dev_trained"])
print("WINNER (best mean dev AUC over the 5 trained checks):", WINNER, {n: round(RES[n]["mean_auc_dev_trained"], 3) for n in VARIANTS})
print("NOTE: all rows are Claude-written and Claude-labelled; the test set is unseen industries/personas but NOT independent.\n")

for n in VARIANTS:
    R, B = RES[n], RES["base"]
    print(f"=== {n}{'  <- winner' if n == WINNER else ''}")
    print(f"{'check':<18}{'n':>4}{'base BA':>9}{'BA':>7}{'95% interval':>16}{'AUC':>7}{'base AUC':>10}{'tuned cut':>11}")
    for c in TRAINED + REGRESSION:
        r, b = R["checks"][c], B["checks"][c]
        if c in TRAINED:
            lo, hi = r["ba_test_tuned_ci"]
            print(f"{c:<18}{r['n_test']:>4}{b['ba_test_frozen_cut']:>9.3f}{r['ba_test_tuned']:>7.3f}  [{lo:.2f}, {hi:.2f}]{r['auc_test']:>9.3f}{b['auc_test']:>10.3f}{r['tuned_cut']:>11.2f}")
        else:
            lo, hi = r["ba_test_frozen_ci"]
            print(f"{c:<18}{r['n_test']:>4}{b['ba_test_frozen_cut']:>9.3f}{r['ba_test_frozen_cut']:>7.3f}  [{lo:.2f}, {hi:.2f}]{r['auc_test']:>9.3f}{b['auc_test']:>10.3f}{'(frozen)':>11}")
    print(f"mean test BA over trained checks: {R['mean_ba_test_trained']:.3f} | ECE test: {R['ece_test']:.3f} (base {B['ece_test']:.3f})")
    d = R["replay_dev"]
    print(f"replay drift on custom-criteria questions (dev, {d['items']} items): mean |change| {d['mean_abs_diff']:.3f}, "
          f"same verdict as base {d['verdict_agreement'] * 100:.0f}%\n")
print("BA for 'base' uses the production cutoff; for trained checks the tuned model uses a cutoff chosen on DEV, so the two columns are not on identical footing.")

In [ ]:
# CELL 11: save to Drive (temp folder -> verify -> rename), prove the Drive copy loads, print the summary to paste back.
# EXPECT: "copied final-<name>: N files / X MB verified" per variant, "Drive copy loads; max diff ... (must be < 1e-4)",
# then a block starting "=== PASTE THIS BACK TO CLAUDE ===".
from importlib.metadata import version

for n in VARIANTS:
    src = f"{WORK}/ckpt-{n}"
    wait_stable(src, quiet_s=5, ignore=("checkpoint_latest",))
    files, size = copy_verified(src, f"{DRIVE_RUN}/final-{n}", f"final-{n}", ignore=("checkpoint_latest",))
    assert sha256_file(f"{src}/model.safetensors") == sha256_file(f"{DRIVE_RUN}/final-{n}/model.safetensors")
    print(f"copied final-{n}: {files} files / {size / 1e6:.0f} MB verified")

os.makedirs(f"{DRIVE_RUN}/files", exist_ok=True)
for n in VARIANTS:
    for fn in (f"preds_{n}.json", f"train_{n}.log"):
        shutil.copy(f"{WORK}/{fn}", f"{DRIVE_RUN}/files/{fn}")
shutil.copy(f"{WORK}/preds_base.json", f"{DRIVE_RUN}/files/preds_base.json")
for split in ("train_all", "dev_all", "test"):
    shutil.copy(f"{DATA_DIR}/{split}.jsonl", f"{DRIVE_RUN}/files/{split}.jsonl")
json.dump(RES, open(f"{DRIVE_RUN}/files/results.json", "w"), indent=1)

manifest = {
    "run_id": RUN_ID, "created": datetime.datetime.now().isoformat(timespec="seconds"), "winner": WINNER,
    "gpu": subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip(),
    "versions": {"python": sys.version.split()[0], "torch": version("torch"), "transformers": version("transformers"),
                 "laya": version("laya"), "laya_commit": LAYA_COMMIT},
    "base": {"repo": BASE_REPO, "revision": BASE_REV, "sha256": BASE_SHA256},
    "frozen_checks_sha256": FROZEN["sha256"], "inputs_sha256": FROZEN["inputs_sha256"],
    "qa_analysis_commit": FROZEN["qa_analysis_commit"], "data_sha256": DATA_SHA256,
    "authorship": "All rows written and labelled by Claude; none human-labelled; test not independent.",
    "training": TRAIN_INFO,
    "final_weights_sha256": {n: sha256_file(f"{DRIVE_RUN}/final-{n}/model.safetensors") for n in VARIANTS},
}
json.dump(manifest, open(f"{DRIVE_RUN}/manifest.json", "w"), indent=2)
os.makedirs(f"{DRIVE}/runs/{RUN_ID}", exist_ok=True)
shutil.copy(f"{DRIVE_RUN}/manifest.json", f"{DRIVE}/runs/{RUN_ID}/manifest.json")
shutil.copy(f"{DRIVE_RUN}/files/results.json", f"{DRIVE}/runs/{RUN_ID}/results.json")

# the Drive copy of the winner must predict what /content predicted
chk = laya.load(f"{DRIVE_RUN}/final-{WINNER}")
sample = DEV[:40]
P = predict_rows(chk, sample)
del chk
torch.cuda.empty_cache()
dmax = max(abs(P[k] - PRED[WINNER]["dev"][k]) for k in P)
print(f"Drive copy loads; max diff vs /content on 40 dev rows: {dmax:.6f}")
assert dmax < 1e-4, "Drive copy predicts differently: stop and tell Claude"

# compact summary
lines = [f"=== PASTE THIS BACK TO CLAUDE ===", f"run {RUN_ID} | winner {WINNER} | gpu {manifest['gpu']}",
         "rows: Claude-written and Claude-labelled (not independent). test = unseen industries/personas."]
for n in VARIANTS:
    lines.append(f"[{n}] wall {TRAIN_INFO[n]['wall_s']}s | losses: " + " ".join(l.split("mean loss")[-1].strip().split()[0] for l in TRAIN_INFO[n]["losses"]))
for c in TRAINED + REGRESSION:
    b = RES["base"]["checks"][c]
    parts = [f"{c:<16} base BA {b['ba_test_frozen_cut']:.2f} AUC {b['auc_test']:.2f}"]
    for n in VARIANTS:
        r = RES[n]["checks"][c]
        if c in TRAINED:
            lo, hi = r["ba_test_tuned_ci"]
            parts.append(f"{n} BA {r['ba_test_tuned']:.2f} [{lo:.2f},{hi:.2f}] AUC {r['auc_test']:.2f} cut {r['tuned_cut']:.2f}")
        else:
            parts.append(f"{n} BA {r['ba_test_frozen_cut']:.2f} AUC {r['auc_test']:.2f}")
    lines.append(" | ".join(parts))
for n in VARIANTS:
    d = RES[n]["replay_dev"]
    lines.append(f"[{n}] ECE {RES[n]['ece_test']:.3f} (base {RES['base']['ece_test']:.3f}) | replay same-verdict {d['verdict_agreement'] * 100:.0f}% mean change {d['mean_abs_diff']:.3f}")
lines.append(f"checkpoints on Drive: MyDrive/laya-finetuning/checkpoints/{RUN_ID}/final-{WINNER} (download this folder)")
SUMMARY = "\n".join(lines)
open(f"{DRIVE}/runs/{RUN_ID}/summary.txt", "w").write(SUMMARY + "\n")
print("\n" + SUMMARY)